# 베이스 코호트 기반 약물 피처 추출

**목적**: `final_df.csv`의 `stay_id`를 기반으로 MIMIC-IV 약물 데이터에서 약물 피처 10개 추출

**수정됨**: final_df.csv 실제 컬럼 구조에 맞게 조정

**출력**: `drug_features.csv` (stay_id별 약물 피처)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# 시각화 설정
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print(" 라이브러리 로드 완료")

 라이브러리 로드 완료


In [2]:
# ============================================================================
# 경로 설정
# ============================================================================

MIMIC4_PATH = "icu"        # MIMIC-IV 데이터 경로
INPUT_CSV = "base_cohort.csv" # 베이스 코호트
OUTPUT_DIR = "output"      # 출력 디렉토리

Path(OUTPUT_DIR).mkdir(exist_ok=True)

print("=" * 80)
print("[설정] 경로 및 디렉토리")
print("=" * 80)
print(f" Input CSV: {INPUT_CSV}")
print(f" MIMIC4 Path: {MIMIC4_PATH}")
print(f" Output Dir: {OUTPUT_DIR}")
print()

[설정] 경로 및 디렉토리
 Input CSV: base_cohort.csv
 MIMIC4 Path: icu
 Output Dir: output



## STEP 1: 베이스 코호트 확인 및 전처리

In [4]:
print("=" * 80)
print("[STEP 1] 베이스 코호트 로드 및 전처리")
print("=" * 80)

try:
    cohort_df = pd.read_csv(INPUT_CSV)
    print(f" 로드 완료: {cohort_df.shape[0]:,} 환자")
    print(f" 원본 컬럼: {list(cohort_df.columns)}")
    print()
    
    # stay_id 추출
    stay_ids = cohort_df['stay_id'].unique()
    print(f" 추출된 stay_id: {len(stay_ids):,}개")
    print(f" stay_id 범위: {stay_ids.min()} ~ {stay_ids.max()}")
    print()
    
    # 필요한 데이터만 선택
    # stay_id만 있으면 됨 (나머지는 inputevents와 병합할 때 필요함)
    cohort_info = cohort_df[['stay_id']].copy()
    
    print(f" 준비 완료")
    print()
    
except Exception as e:
    print(f" 오류: {e}")
    raise

[STEP 1] 베이스 코호트 로드 및 전처리
 로드 완료: 39,829 환자
 원본 컬럼: ['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los', 'icu_los_days', 'win_start', 'win_end', 'log_los', 'pre_icu_los', 'race_simplified']

 추출된 stay_id: 39,829개
 stay_id 범위: 30000646 ~ 39999858

 준비 완료



## STEP 2: MIMIC-IV inputevents 로드 및 필터링

In [8]:
print("=" * 80)
print("[STEP 2] 약물 데이터(inputevents) 로드")
print("=" * 80)

try:
    # inputevents 필터링 (stay_id 기반)
    print(f" ICU inputevents 로드 중...")
    inputevents = pd.read_csv(
        f"{MIMIC4_PATH}/icu_inputevents.csv",
        usecols=['stay_id', 'itemid', 'rate', 'amount', 'starttime', 'endtime'],
        dtype={'stay_id': 'int64', 'rate': 'float64', 'amount': 'float64'}
    )
    
    print(f" 로드 완료: {len(inputevents):,} 약물 투여 기록")
    
    # 날짜 변환
    inputevents['starttime'] = pd.to_datetime(inputevents['starttime'])
    inputevents['endtime'] = pd.to_datetime(inputevents['endtime'])
    
    # 코호트의 stay_id만 필터링
    inputevents_filtered = inputevents[inputevents['stay_id'].isin(stay_ids)].copy()
    print(f" 필터링 완료: {len(inputevents_filtered):,} 레코드 (코호트에 해당)")
    print(f"  - 포함된 stay_id: {inputevents_filtered['stay_id'].nunique():,}개")
    
    inputevents = inputevents_filtered
    print(f" 준비 완료")
    print()
    
except Exception as e:
    print(f" 오류: {e}")
    raise

[STEP 2] 약물 데이터(inputevents) 로드
 ICU inputevents 로드 중...
 로드 완료: 10,953,713 약물 투여 기록
 필터링 완료: 5,392,132 레코드 (코호트에 해당)
  - 포함된 stay_id: 39,829개
 준비 완료



## STEP 3: 약물 분류 정의

In [6]:
print("=" * 80)
print("[STEP 3] 약물 분류 정의")
print("=" * 80)

# 약물 분류 (itemid 기반)
VASOACTIVE_DRUGS = {221662, 221653, 221289, 221749, 221309, 221906, 221833}
ANALGOSEDATION = {222038, 225943, 225942, 225940, 225941, 222042}
DIURETICS = {221407, 221479, 222033}
INSULIN = {223257, 223258, 223259, 223260}

print(f" 혈관활성약물 (Vasoactive): {len(VASOACTIVE_DRUGS)}개")
print(f" 진통진정제 (Analgo-sedation): {len(ANALGOSEDATION)}개")
print(f" 이뇨제 (Diuretics): {len(DIURETICS)}개")
print(f" 인슐린 (Insulin): {len(INSULIN)}개")
print()

[STEP 3] 약물 분류 정의
 혈관활성약물 (Vasoactive): 7개
 진통진정제 (Analgo-sedation): 6개
 이뇨제 (Diuretics): 3개
 인슐린 (Insulin): 4개



## STEP 4: 특징 추출 함수 정의

In [7]:
def extract_drug_features_for_stay(stay_id, inputevents_df):
    """
    개별 stay에 대해 48시간 윈도우 내 약물 피처 10개 추출
    
    주의: 이 함수는 ICU 입실부터 48시간 동안의 약물 데이터를 사용합니다.
    정확한 intime이 필요하면 admissions, icustays 테이블과 병합 필요합니다.
    """
    
    # 해당 stay의 약물 데이터
    stay_drugs = inputevents_df[inputevents_df['stay_id'] == stay_id].copy()
    
    # 약물이 없으면 0 반환
    if len(stay_drugs) == 0:
        return {f: 0 for f in [
            'overall_rate_cv', 'analgosedation_intensity', 'vasoactive_load',
            'total_sedation_exposure', 'insulin_requirement', 'drug_diversity_index',
            'drug_class_ratio', 'concurrent_intensity', 'critical_medication_score',
            'medication_escalation'
        ]}
    
    # 48시간 윈도우 정의
    win_start = stay_drugs['starttime'].min()
    win_end = win_start + pd.Timedelta(hours=48)
    
    # 48시간 내 약물만 필터링
    stay_drugs = stay_drugs[
        (stay_drugs['starttime'] <= win_end) &
        (stay_drugs['endtime'] >= win_start)
    ].copy()
    
    if len(stay_drugs) == 0:
        return {f: 0 for f in [
            'overall_rate_cv', 'analgosedation_intensity', 'vasoactive_load',
            'total_sedation_exposure', 'insulin_requirement', 'drug_diversity_index',
            'drug_class_ratio', 'concurrent_intensity', 'critical_medication_score',
            'medication_escalation'
        ]}
    
    features = {}
    
    # 1. 투여율 변동계수
    rates = stay_drugs['rate'].dropna()
    features['overall_rate_cv'] = rates.std() / (rates.mean() + 1e-6) if len(rates) > 1 else 0
    
    # 2. 진통진정제 강도
    analgo_data = stay_drugs[stay_drugs['itemid'].isin(ANALGOSEDATION)]
    analgo_dose = analgo_data['amount'].sum()
    analgo_duration = (analgo_data['endtime'].max() - analgo_data['starttime'].min()).total_seconds() / 3600 if len(analgo_data) > 0 else 0
    features['analgosedation_intensity'] = analgo_dose / (analgo_duration + 1)
    
    # 3. 혈관활성약물 부하
    vaso_data = stay_drugs[stay_drugs['itemid'].isin(VASOACTIVE_DRUGS)]
    vaso_duration = (vaso_data['endtime'].max() - vaso_data['starttime'].min()).total_seconds() / 3600 if len(vaso_data) > 0 else 0
    features['vasoactive_load'] = (1 if len(vaso_data) > 0 else 0) * vaso_duration
    
    # 4. 진정 노출도
    diuretic_data = stay_drugs[stay_drugs['itemid'].isin(DIURETICS)]
    diuretic_dose = diuretic_data['amount'].sum()
    features['total_sedation_exposure'] = analgo_dose * 1.0 + diuretic_dose * 0.5
    
    # 5. 인슐린 필요량
    insulin_data = stay_drugs[stay_drugs['itemid'].isin(INSULIN)]
    features['insulin_requirement'] = insulin_data['amount'].sum()
    
    # 6-7. 약물 다양성 및 분류 비율
    unique_drugs = stay_drugs['itemid'].nunique()
    max_concurrent = 1
    for idx, row in stay_drugs.sort_values('starttime').iterrows():
        concurrent = len(stay_drugs[
            (stay_drugs['starttime'] <= row['endtime']) & 
            (stay_drugs['endtime'] >= row['starttime'])
        ])
        max_concurrent = max(max_concurrent, concurrent)
    
    features['drug_diversity_index'] = unique_drugs / (max_concurrent + 1)
    features['drug_class_ratio'] = min(unique_drugs, 5) / (unique_drugs + 1)
    
    # 8. 동시투여 강도
    features['concurrent_intensity'] = max_concurrent
    
    # 9. 중증 약물 점수
    score = (
        (1 if len(vaso_data) > 0 else 0) * 3 +
        (1 if len(analgo_data) > 0 else 0) * 2 +
        (1 if len(diuretic_data) > 0 else 0) * 1
    )
    features['critical_medication_score'] = score
    
    # 10. 약물 에스컬레이션
    early_window = win_start + pd.Timedelta(hours=24)
    early_drugs = len(stay_drugs[stay_drugs['starttime'] < early_window])
    late_drugs = len(stay_drugs[stay_drugs['starttime'] >= early_window])
    features['medication_escalation'] = (late_drugs - early_drugs) / (early_drugs + 1)
    
    return features

print("=" * 80)
print("[STEP 4] 특징 추출 함수 정의")
print("=" * 80)
print(" 함수 정의 완료")
print()
print("추출될 10개 피처:")
for i, feat in enumerate([
    'overall_rate_cv', 'analgosedation_intensity', 'vasoactive_load',
    'total_sedation_exposure', 'insulin_requirement', 'drug_diversity_index',
    'drug_class_ratio', 'concurrent_intensity', 'critical_medication_score',
    'medication_escalation'
], 1):
    print(f" {i:2}. {feat}")
print()

[STEP 4] 특징 추출 함수 정의
 함수 정의 완료

추출될 10개 피처:
  1. overall_rate_cv
  2. analgosedation_intensity
  3. vasoactive_load
  4. total_sedation_exposure
  5. insulin_requirement
  6. drug_diversity_index
  7. drug_class_ratio
  8. concurrent_intensity
  9. critical_medication_score
 10. medication_escalation



## STEP 5: 약물 피처 추출 (모든 환자)

In [ ]:
print("=" * 80)
print("[STEP 5] 약물 피처 추출 (진행 중...)")
print("=" * 80)

feature_list = []
total_stays = len(stay_ids)

for i, stay_id in enumerate(stay_ids):
    if (i + 1) % 1000 == 0 or (i + 1) == total_stays:
        print(f"  Progress: {i + 1:,} / {total_stays:,}", end='\r')
    
    try:
        features = extract_drug_features_for_stay(stay_id, inputevents)
        feature_list.append(features)
    except Exception as e:
        print(f"\n   경고 (stay_id={stay_id}): {e}")
        feature_list.append({f: 0 for f in [
            'overall_rate_cv', 'analgosedation_intensity', 'vasoactive_load',
            'total_sedation_exposure', 'insulin_requirement', 'drug_diversity_index',
            'drug_class_ratio', 'concurrent_intensity', 'critical_medication_score',
            'medication_escalation'
        ]})

features_df = pd.DataFrame(feature_list)
print(f"\n 추출 완료: {len(features_df):,} 환자의 약물 피처")
print()

## STEP 6: 최종 데이터 병합

In [8]:
print("=" * 80)
print("[STEP 6] 최종 데이터 병합")
print("=" * 80)

# stay_id와 피처 결합
final_drug_features = pd.DataFrame({'stay_id': stay_ids}).reset_index(drop=True)
final_drug_features = pd.concat([final_drug_features, features_df.reset_index(drop=True)], axis=1)


print(f" 최종 데이터 생성")
print(f"  - Shape: {final_drug_features.shape}")
print(f"  - 환자: {len(final_drug_features):,}")
print(f"  - 컬럼: {final_drug_features.shape[1]}")
print(f"  - 컬럼명: {list(final_drug_features.columns)}")

# CSV 저장
output_csv = f"{OUTPUT_DIR}/drug_features.csv"
final_drug_features.to_csv(output_csv, index=False)
print(f"\n CSV 저장: {output_csv}")
print(f"  - 파일 크기: {open(output_csv).read().__len__() / 1024 / 1024:.1f} MB")
print()

[STEP 6] 최종 데이터 병합
✓ 최종 데이터 생성
  - Shape: (16394, 11)
  - 환자: 16,394
  - 컬럼: 11
  - 컬럼명: ['stay_id', 'overall_rate_cv', 'analgosedation_intensity', 'vasoactive_load', 'total_sedation_exposure', 'insulin_requirement', 'drug_diversity_index', 'drug_class_ratio', 'concurrent_intensity', 'critical_medication_score', 'medication_escalation']

✓ CSV 저장: output/drug_features.csv
  - 파일 크기: 1.9 MB



## STEP 7: 약물 피처 기본 통계

In [9]:
print("=" * 80)
print("[STEP 7] 약물 피처 기본 통계")
print("=" * 80)

feature_cols = [
    'overall_rate_cv', 'analgosedation_intensity', 'vasoactive_load',
    'total_sedation_exposure', 'insulin_requirement', 'drug_diversity_index',
    'drug_class_ratio', 'concurrent_intensity', 'critical_medication_score',
    'medication_escalation'
]

print(final_drug_features[feature_cols].describe())
print()
print("누락된 값 확인:")
print(final_drug_features[feature_cols].isnull().sum())

[STEP 7] 약물 피처 기본 통계
       overall_rate_cv  analgosedation_intensity  vasoactive_load  \
count     16394.000000              16394.000000     16394.000000   
mean          1.571554                 10.517632        12.053269   
std           0.993718                 17.536331        17.649034   
min           0.000000                  0.000000         0.000000   
25%           0.933327                  0.000000         0.000000   
50%           1.433207                  5.766388         0.000000   
75%           2.168374                 15.389631        23.291667   
max          11.286768                693.641618        92.883333   

       total_sedation_exposure  insulin_requirement  drug_diversity_index  \
count             16394.000000         16394.000000          16394.000000   
mean                259.619046            21.909544              0.534326   
std                 389.460916            82.648227              0.383253   
min                   0.000000             0.0000

## STEP 8: 시각화

In [12]:
print("\n" + "=" * 80)
print(" 모든 작업 완료!")
print("=" * 80)
print(f"\n 출력 파일:")
print(f"  - {output_csv}")


✅ 모든 작업 완료!

📊 출력 파일:
  - output/drug_features.csv
  - output/drug_features_distributions.png
  - output/drug_features_correlation.png



In [13]:
print("[검증 1] 기본 정보")
print(f"Shape: {final_drug_features.shape}")
print(f"Columns: {list(final_drug_features.columns)}")
print()

print("[검증 2] 상위 5개 행")
print(final_drug_features.head())
print()

print("[검증 3] 피처별 통계")
print(final_drug_features[feature_cols].describe())

[검증 1] 기본 정보
Shape: (16394, 11)
Columns: ['stay_id', 'overall_rate_cv', 'analgosedation_intensity', 'vasoactive_load', 'total_sedation_exposure', 'insulin_requirement', 'drug_diversity_index', 'drug_class_ratio', 'concurrent_intensity', 'critical_medication_score', 'medication_escalation']

[검증 2] 상위 5개 행
    stay_id  overall_rate_cv  analgosedation_intensity  vasoactive_load  \
0  30001336         1.292733                       0.0              0.0   
1  30001471         1.273145                       0.0              0.0   
2  30003087         2.661236                       0.0              0.0   
3  30004306         0.001168                       0.0              0.0   
4  30004462         0.551107                       0.0              0.0   

   total_sedation_exposure  insulin_requirement  drug_diversity_index  \
0                      0.0                  0.0              0.350000   
1                      0.0                  0.0              0.500000   
2                      